In [2]:
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llama3.2")

#--Problem without memory 
print("Without memory - LLM forgets everything : \n")

response1 = chat.invoke("I am Ankit and I work as data scientist")
print(f"Turn 1: {response1.content}\n")

response2 = chat.invoke("What is my name?")
print(f"Turn 2: {response2.content}\n")

response3 = chat.invoke("What do I do for work?")
print(f"Turn 3: {response3.content}\n")

print("─" * 60)
print("It has no idea. Every call is stateless.")
print("This is why memory exists.")


Without memory - LLM forgets everything : 

Turn 1: Hello Ankit! It's nice to meet you. As a data scientist, I'm sure you're always looking for ways to improve your skills and stay up-to-date with the latest developments in the field.

What area of data science are you currently working on? Are you working on any exciting projects or looking to explore new areas such as machine learning, natural language processing, or data visualization?

Turn 2: I don't have any information about your personal identity, and I'm not capable of knowing your name. This conversation just started, and I'm here to help answer any questions you may have. Would you like to tell me your name, or is there something else I can assist you with?

Turn 3: I'd be happy to help you explore your career options. However, I need a bit more information from you.

Could you please tell me:

1. What are your hobbies or interests?
2. Do you have any relevant education or training?
3. Are there any specific industries or jo

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat = ChatOllama(model="llama3.2")

#--Manual memory

chat_history = []

def chat_with_memory(user_input: str) -> str:
    # Add user message to history
    chat_history.append(HumanMessage(content=user_input))

    # Pass full history to the model
    response = chat.invoke(chat_history)

    # Add model response to history
    chat_history.append(AIMessage(content=response.content))

    return response.content

# ── Have a multi-turn conversation ────────────────────────────────────────
conversations = [
    "My name is Ankit and I am a manager learning ML.",
    "I have built 6 projects so far including a Bayesian A/B tester and a Monte Carlo simulator.",
    "What is my name?",
    "What projects have I built?",
    "Based on what I told you, what should I learn next?",
]


for turn, message in enumerate(conversations, 1):
    response = chat_with_memory(message)
    print(f"Turn {turn}")
    print(f"Human: {message}")
    print(f"AI:    {response}\n")
    print("─" * 60 + "\n")

print(f"Total messages in history: {len(chat_history)}")
print("History contains:")
for msg in chat_history:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"  {role}: {msg.content[:60]}...")



Turn 1
Human: My name is Ankit and I am a manager learning ML.
AI:    Hi Ankit, nice to meet you! Congratulations on taking the first step towards learning Machine Learning (ML). As a manager, it's great that you're interested in acquiring this skill to enhance your leadership capabilities.

Machine Learning can be a valuable asset for any organization, and having a basic understanding of ML concepts can help you make informed decisions about data-driven projects. What specific areas of ML are you interested in learning more about? Are you looking to improve your understanding of supervised/unsupervised learning, deep learning, or perhaps explore applications like natural language processing (NLP) or computer vision?

Feel free to share any questions or topics that interest you, and I'll do my best to guide you on this exciting journey!

────────────────────────────────────────────────────────────

Turn 2
Human: I have built 6 projects so far including a Bayesian A/B tester and a Monte

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

chat   = ChatOllama(model="llama3.2")
parser = StrOutputParser()

# ChatPromptTemplate with MessagesPlaceholder
# MessagePlaceholder is a slot where full chat history gets inserted

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful ML tutor. You remember everything the student tells you."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | chat | parser

# Conversation manager
class ConversationManager:
    def __init__(self):
        self.history = []

    def chat(self, user_input: str) -> str:
        response = chain.invoke({
            "chat_history" : self.history,
            "input" : user_input
        })

        # Update history
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=response))
        return response
    
    def reset(self):
        self.history = []
        print("Conversation reset")

    def show_history(self):
        print(f"\nConversation history ({len(self.history)} messages):")
        for msg in self.history:
            role = "Human" if isinstance(msg, HumanMessage) else "AI"
            print(f"  {role}: {msg.content[:80]}...")


# -- Test conversation manager
tutor = ConversationManager()

turns = [
    "I am Ankit. I know Python and have built projects in Bayesian statistics and Monte Carlo simulation.",
    "I want to learn NLP next. What should I start with?",
    "What Python libraries will I need for that?",
    "Given what you know about my background, how long do you think it will take me?",
]


for user_input in turns:
    print(f"Ankit: {user_input}")
    response = tutor.chat(user_input)
    print(f"Tutor: {response}\n")
    print("─" * 60 + "\n")

tutor.show_history()


Ankit: I am Ankit. I know Python and have built projects in Bayesian statistics and Monte Carlo simulation.
Tutor: Nice to meet you, Ankit! I've taken note of your background in Python, Bayesian statistics, and Monte Carlo simulation. That's a great combination!

It sounds like you have a solid foundation in statistical modeling and simulation techniques. What specific area within Bayesian statistics are you interested in exploring further? Are you looking to dive deeper into topics like Markov Chain Monte Carlo (MCMC), Bayesian inference, or something else?

Also, since you've worked with Python, I'm assuming you're familiar with libraries such as NumPy, pandas, and SciPy. Have you explored any other libraries like Scikit-learn for machine learning, or PyMC3/Metropolis for more advanced Bayesian modeling?

────────────────────────────────────────────────────────────

Ankit: I want to learn NLP next. What should I start with?
Tutor: NLP (Natural Language Processing) is a fascinating fi